# Method comparison (SI Figure 8)

This notebook fits XenoComm and runs CellChat and LIANA on the distributed PDX subset and native 10x dataset, then recreates the detection-rate heatmaps, native ligand counts, and top-k overlaps. Install the comparison dependencies and CellChat using the commands in the repository README before running it.

The distributed PDX H5AD inputs (10,000 cells per species) are in `data/melanoma_pdx_10k`; the 10x count matrix downloads automatically. This analysis uses the same database-restricted receptor–target model as the other notebooks and five matched comparator seeds. It runs from scratch without outputs from any other notebook.

PDX expression is rebuilt from raw counts in `.raw.X`: normalize each cell to 10,000 counts, then apply natural-log1p. The original counts are retained in `layers["counts"]`; saved expression and gene-selection statistics are not used.


In [ ]:
import gc
import os
import subprocess
from pathlib import Path
from shutil import copyfileobj
from urllib.request import Request, urlopen

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import tensorflow as tf
import xenocomm as xc
from scipy import sparse
from xenocomm import comparison
from xenocomm._download import ensure_database

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "data").is_dir():
    NOTEBOOK_DIR = NOTEBOOK_DIR / "notebooks"

sns.set_theme(context="notebook", style="whitegrid")
OUTPUT_DIR = NOTEBOOK_DIR / "outputs/comparison_10k"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REUSE_OUTPUTS = False

SEEDS = (20260716, 20260717, 20260718, 20260719, 20260720)
CELLCHAT_NBOOT = 100
LIANA_N_PERMS = 1000
LIANA_EXPR_PROP = 0.1
LIANA_MIN_CELLS = 20
SIGNIFICANCE_THRESHOLD = 0.05
TOP_K = (10, 25, 50)
POSTERIOR_SAMPLES = 2000

R_HOME = subprocess.check_output(["R", "RHOME"], text=True).strip()
R_LIBRARIES = (
    subprocess.check_output(["Rscript", "-e", 'cat(.libPaths(), sep="\n")'], text=True)
    .strip()
    .splitlines()
)
R_LIBRARIES = os.pathsep.join(R_LIBRARIES)
cellchat_database, cellchat_genes = comparison.cellchat_resources(R_HOME, R_LIBRARIES)
import liana as li

liana_resource = li.resource.select_resource("cellphonedb")
ortholog_path = ensure_database() / "human_mouse_orthologs.parquet"
print(f"CellChat database: {len(cellchat_database):,} interactions")
print(f"LIANA resource: {len(liana_resource):,} interactions")

In [ ]:
PDX_DIR = NOTEBOOK_DIR / "data/melanoma_pdx_10k"
for filename in ("adata_human.h5ad", "adata_mouse.h5ad"):
    if not (PDX_DIR / filename).is_file():
        raise FileNotFoundError(f"Place the PDX input file at {PDX_DIR / filename}")
def read_pdx_counts(path):
    stored = ad.read_h5ad(path)
    if stored.raw is None:
        raise ValueError(f"{path} must contain raw counts in .raw.X")
    counts = sparse.csr_matrix(stored.raw.X, copy=True)
    if (
        not np.isfinite(counts.data).all()
        or np.any(counts.data < 0)
        or np.any(counts.data != np.floor(counts.data))
    ):
        raise ValueError(f"{path}: .raw.X must contain nonnegative integer counts")
    if np.any(np.asarray(counts.sum(axis=1)).ravel() <= 0):
        raise ValueError(f"{path}: every cell must have a positive count total")
    data = ad.AnnData(
        X=counts.astype(np.float32),
        obs=stored.obs[["sample", "cell_type"]].copy(),
        var=stored.raw.var[["gene_id"]].copy(),
    )
    data.layers["counts"] = counts
    sc.pp.normalize_total(data, target_sum=10_000)
    sc.pp.log1p(data)
    return data


pdx_human = read_pdx_counts(PDX_DIR / "adata_human.h5ad")
pdx_mouse = read_pdx_counts(PDX_DIR / "adata_mouse.h5ad")

TENX_DIR = NOTEBOOK_DIR / "data/10x_hgmm"
TENX_DIR.mkdir(parents=True, exist_ok=True)
INPUT_H5 = TENX_DIR / "10k_hgmm_3p_gemx_count_sample_filtered_feature_bc_matrix.h5"
DATA_URL = (
    "https://cf.10xgenomics.com/samples/cell-exp/8.0.0/"
    "10k_hgmm_3p_gemx_10k_hgmm_3p_gemx/"
    "10k_hgmm_3p_gemx_10k_hgmm_3p_gemx_count_sample_filtered_feature_bc_matrix.h5"
)
if not INPUT_H5.exists():
    download = INPUT_H5.with_suffix(".download")
    request = Request(
        DATA_URL, headers={"Range": "bytes=0-", "User-Agent": "Mozilla/5.0"}
    )
    with urlopen(request) as source, download.open("wb") as destination:
        copyfileobj(source, destination)
    download.replace(INPUT_H5)
raw = sc.read_10x_h5(INPUT_H5)

In [ ]:
human_genes = raw.var["genome"].eq("GRCh38").to_numpy()
mouse_genes = raw.var["genome"].eq("GRCm39").to_numpy()
human_umis = np.asarray(raw[:, human_genes].X.sum(axis=1)).ravel()
mouse_umis = np.asarray(raw[:, mouse_genes].X.sum(axis=1)).ravel()
total_umis = human_umis + mouse_umis
human_cells = human_umis / total_umis >= 0.9
mouse_cells = mouse_umis / total_umis >= 0.9


def collapse_species(cell_mask, gene_mask, prefix):
    subset = raw[cell_mask, gene_mask]
    symbols = pd.Index(subset.var_names.str.removeprefix(prefix), name="gene_symbol")
    gene_ids = subset.var["gene_ids"].astype(str).str.removeprefix(prefix)
    unique_symbols = symbols.drop_duplicates()
    output_index = pd.Series(np.arange(len(unique_symbols)), index=unique_symbols)
    columns = output_index.loc[symbols].to_numpy()
    projection = sparse.csr_matrix(
        (np.ones(len(symbols), dtype=np.int32), (np.arange(len(symbols)), columns)),
        shape=(len(symbols), len(unique_symbols)),
    )
    counts = (subset.X.tocsr().astype(np.int32) @ projection).tocsr()
    ids = (
        pd.Series(gene_ids.to_numpy(), index=symbols)
        .groupby(level=0, sort=False)
        .agg(";".join)
    )
    var = pd.DataFrame(
        {"gene_ids": ids.loc[unique_symbols].to_numpy()}, index=unique_symbols
    )
    return counts, var, subset.obs_names.copy()


collapsed = {
    "human": collapse_species(human_cells, human_genes, "GRCh38_"),
    "mouse": collapse_species(mouse_cells, mouse_genes, "GRCm39_"),
}
shared_genes = sorted(
    {gene.casefold() for gene in collapsed["human"][1].index}
    & {gene.casefold() for gene in collapsed["mouse"][1].index}
)


def normalize_species(species, cell_line):
    counts, var, obs_names = collapsed[species]
    lookup = {gene.casefold(): index for index, gene in enumerate(var.index)}
    shared_indices = np.asarray([lookup[gene] for gene in shared_genes])
    library_size = np.asarray(counts[:, shared_indices].sum(axis=1)).ravel()
    expression = (
        counts.astype(np.float32).multiply((10_000 / library_size)[:, None]).tocsr()
    )
    np.log1p(expression.data, out=expression.data)
    obs = pd.DataFrame({"cell_line": cell_line, "species": species}, index=obs_names)
    result = ad.AnnData(expression, obs=obs, var=var)
    result.layers["counts"] = counts
    if species == "mouse":
        dispersion_input = ad.AnnData(
            counts[:, shared_indices].copy(), var=var.iloc[shared_indices].copy()
        )
        sc.pp.log1p(dispersion_input)
        sc.pp.highly_variable_genes(dispersion_input)
        dispersions = pd.Series(
            dispersion_input.var["dispersions_norm"].to_numpy(),
            index=dispersion_input.var_names.str.casefold(),
        )
        result.var["dispersions_norm"] = [
            dispersions.get(gene.casefold(), np.nan) for gene in var.index
        ]
    return result


tenx_human = normalize_species("human", "HEK293T")
tenx_mouse = normalize_species("mouse", "NIH3T3")
print(f"Human: {tenx_human.n_obs:,} cells; mouse: {tenx_mouse.n_obs:,} cells")
print(f"Mixed or ambiguous barcodes removed: {(~human_cells & ~mouse_cells).sum():,}")

In [ ]:
def validation_indices(obs, size, seed, stratify):
    rng = np.random.RandomState(seed)
    if stratify:
        strata = obs[["sample", "cell_type"]].astype(str).agg("|".join, axis=1)
        counts = strata.value_counts().sort_index()
        exact = counts.to_numpy() * size / len(obs)
        quotas = np.floor(exact).astype(int)
        quotas[np.argsort(-(exact - quotas), kind="stable")[: size - quotas.sum()]] += 1
        held_out = np.concatenate(
            [
                rng.choice(
                    np.flatnonzero(strata.to_numpy() == label), quota, replace=False
                )
                for label, quota in zip(counts.index, quotas, strict=True)
            ]
        )
    else:
        held_out = rng.choice(len(obs), size, replace=False)
    rng.shuffle(held_out)
    return np.setdiff1d(np.arange(len(obs)), held_out), held_out


studies = {
    "PDX": {
        "human": pdx_human,
        "mouse": pdx_mouse,
        "batch_size": 1024,
        "epochs": 5,
        "validation_cells": 4096,
        "training_seed": 20260716,
        "posterior_seed": 20260717,
        "split_seed": 20260717,
        "validation_seed": 20260718,
        "cell_cap": 200,
    },
    "10x": {
        "human": tenx_human,
        "mouse": tenx_mouse,
        "batch_size": 687,
        "epochs": 29,
        "validation_cells": 687,
        "training_seed": 20260717,
        "posterior_seed": 20260718,
        "split_seed": 20260718,
        "validation_seed": 20260719,
        "cell_cap": 1200,
    },
}

In [ ]:
for dataset, study in studies.items():
    directory = OUTPUT_DIR / dataset.lower()
    directory.mkdir(parents=True, exist_ok=True)
    table_path = directory / "xenocomm_ligands.parquet"
    if REUSE_OUTPUTS and table_path.is_file():
        study["ligands"] = pd.read_parquet(table_path)
        continue
    mouse, human = study["mouse"], study["human"]
    training_indices, held_out = validation_indices(
        mouse.obs, study["validation_cells"], study["split_seed"], dataset == "PDX"
    )
    network = xc.prepare_network(mouse, human, dispersion_cutoff=-10)
    abundance = xc.compute_ligand_abundance(
        mouse,
        human,
        network["ligands"],
        network["human_ligands"],
        network["ligand_receptor_matrix"],
    )
    model = xc.XenocommModel(
        mouse[training_indices],
        **network,
        mean_ligand=abundance,
        receptor_target_mode="learned",
        training_seed=study["training_seed"],
        posterior_seed=study["posterior_seed"],
        batch_size=study["batch_size"],
        steps_per_batch=250,
        epochs=study["epochs"],
    )
    print(f"Training XenoComm: {dataset}", flush=True)
    model.train(
        validation_mouse=mouse[held_out],
        absolute_tolerance=0.001 * study["batch_size"],
        patience=3,
        min_evaluations=5,
        validation_seed=study["validation_seed"],
    )
    samples = model.sample(POSTERIOR_SAMPLES)
    parameters = model.get_parameters()
    study["ligands"] = xc.ligand_result_table(
        model.ligands,
        samples,
        model.mean_ligand_np,
        model.ligand_receptor_matrix_np,
        xc.get_receptor_sensitivity(parameters),
    )
    np.savez(
        directory / "model.staged.npz",
        **{
            "ligands": np.asarray(model.ligands),
            "human_ligands": np.asarray(model.human_ligands),
            "receptors": np.asarray(model.receptors),
            "targets": np.asarray(model.targets),
            "mean_ligand_np": model.mean_ligand_np,
            "ligand_receptor_matrix_np": model.ligand_receptor_matrix_np,
            **{f"s_{key}": value for key, value in samples.items()},
            **{f"v_{key}": value for key, value in parameters.items()},
        },
    )
    study["ligands"].to_parquet(table_path, index=False)
    del model, samples, parameters
    tf.keras.backend.clear_session()
    gc.collect()

In [ ]:
replicate_tables = []
consensus_tables = []
coverage_rows = []
for dataset, settings in studies.items():
    study = comparison.prepare_comparison(
        dataset,
        settings["human"],
        settings["mouse"],
        settings["ligands"],
        ortholog_path,
        cellchat_database,
        liana_resource,
    )
    directory = OUTPUT_DIR / dataset.lower()
    interaction_tables = []
    for seed in SEEDS:
        path = directory / f"interactions_{seed}.parquet"
        if REUSE_OUTPUTS and path.is_file():
            interactions = pd.read_parquet(path)
        else:
            print(f"Running CellChat and LIANA: {dataset}, seed {seed}", flush=True)
            interactions = comparison.run_comparators(
                study,
                seed,
                cellchat_genes,
                r_home=R_HOME,
                r_libraries=R_LIBRARIES,
                cell_cap=settings["cell_cap"],
                nboot=CELLCHAT_NBOOT,
                n_perms=LIANA_N_PERMS,
                expr_prop=LIANA_EXPR_PROP,
                min_cells=LIANA_MIN_CELLS,
                significance_threshold=SIGNIFICANCE_THRESHOLD,
            )
            interactions.to_parquet(path, index=False)
        interaction_tables.append(interactions)
    interactions = pd.concat(interaction_tables, ignore_index=True)
    scores, replicate, consensus = comparison.rank_ligands(study, interactions, SEEDS)
    interactions.to_parquet(directory / "method_interactions.parquet", index=False)
    scores.to_parquet(directory / "target_ligand_scores.parquet", index=False)
    replicate_tables.append(replicate.assign(dataset=dataset))
    consensus_tables.append(consensus.assign(dataset=dataset))
    for method, native in study.native.items():
        coverage_rows.append(
            {
                "dataset": dataset,
                "method": comparison.METHOD_LABELS[method],
                "native_ligands": len(native),
                "shared_ligands": len(study.shared),
            }
        )

replicate_rankings = pd.concat(replicate_tables, ignore_index=True)
consensus_rankings = pd.concat(consensus_tables, ignore_index=True)
replicate_rankings.to_parquet(OUTPUT_DIR / "replicate_rankings.parquet", index=False)
consensus_rankings.to_parquet(OUTPUT_DIR / "consensus_rankings.parquet", index=False)
coverage = pd.DataFrame(coverage_rows)
coverage.to_parquet(OUTPUT_DIR / "ligand_coverage.parquet", index=False)
display(coverage)

In [ ]:
decile_tables = []
for dataset in ("PDX", "10x"):
    shared = replicate_rankings.loc[
        replicate_rankings.dataset.eq(dataset)
        & replicate_rankings.universe.eq("shared")
    ]
    keys = ["replicate_seed", "replicate", "ligand_id"]
    xenocomm = shared.loc[shared.method.eq("xenocomm"), [*keys, "percentile"]]
    for method in ("cellchat", "liana_cellphonedb"):
        comparator = shared.loc[shared.method.eq(method), [*keys, "detected"]]
        paired = xenocomm.merge(comparator, on=keys, validate="one_to_one")
        if len(paired) != len(xenocomm):
            raise ValueError("Incomplete shared-universe comparison")
        paired["xenocomm_decile"] = (
            paired.percentile.mul(10).clip(upper=9.999999).astype(int) + 1
        )
        decile_tables.append(
            paired.groupby(
                ["replicate_seed", "replicate", "xenocomm_decile"], as_index=False
            )
            .agg(
                detected_fraction=("detected", "mean"),
                ligand_count=("ligand_id", "size"),
            )
            .assign(dataset=dataset, comparator=comparison.METHOD_LABELS[method])
        )
decile_detection = pd.concat(decile_tables, ignore_index=True)

native = consensus_rankings.loc[consensus_rankings.universe.eq("native")].copy()
native["detected_any"] = native.detected_fraction.gt(0)
native_detection_counts = native.groupby(["dataset", "method"], as_index=False).agg(
    detected_ligands=("detected_any", "sum")
)
native_detection_counts["method"] = native_detection_counts.method.map(
    comparison.METHOD_LABELS
)

overlap_rows = []
for dataset, universe in (("PDX", "shared"), ("10x", "native")):
    table = consensus_rankings.loc[
        consensus_rankings.dataset.eq(dataset)
        & consensus_rankings.universe.eq(universe)
    ]
    for method in ("cellchat", "liana_cellphonedb"):
        left = table.loc[table.method.eq("xenocomm")].sort_values("rank")
        right = table.loc[table.method.eq(method)].sort_values("rank")
        for k in TOP_K:
            top_xenocomm = set(left.head(k).ligand_id)
            top_comparator = set(right.head(k).ligand_id)
            union = top_xenocomm | top_comparator
            overlap_rows.append(
                {
                    "dataset": dataset,
                    "universe": universe,
                    "comparator": comparison.METHOD_LABELS[method],
                    "k": k,
                    "xenocomm_k": len(top_xenocomm),
                    "comparator_k": len(top_comparator),
                    "intersection": len(top_xenocomm & top_comparator),
                    "union": len(union),
                    "jaccard": len(top_xenocomm & top_comparator) / len(union)
                    if union
                    else 1.0,
                }
            )
topk_overlap = pd.DataFrame(overlap_rows)
for name, table in (
    ("decile_detection", decile_detection),
    ("native_detection_counts", native_detection_counts),
    ("topk_overlap", topk_overlap),
):
    table.to_parquet(OUTPUT_DIR / f"{name}.parquet", index=False)
display(native_detection_counts)
display(topk_overlap)

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(10, 12), layout="constrained")
heatmap_specs = (
    ("PDX", "CellChat"),
    ("PDX", "LIANA"),
    ("10x", "CellChat"),
    ("10x", "LIANA"),
)
for letter, ax, (dataset, comparator) in zip(
    "abcd", axes[:2].flat, heatmap_specs, strict=True
):
    frame = decile_detection.loc[
        decile_detection.dataset.eq(dataset)
        & decile_detection.comparator.eq(comparator)
    ]
    matrix = frame.pivot(
        index="replicate_seed", columns="xenocomm_decile", values="detected_fraction"
    ).reindex(index=SEEDS, columns=range(1, 11))
    ax.set_facecolor("#dddddd")
    sns.heatmap(
        matrix,
        vmin=0,
        vmax=1,
        cmap="viridis",
        ax=ax,
        yticklabels=[f"S{str(seed)[-2:]}" for seed in SEEDS],
        cbar_kws={"label": "Fraction detected"},
    )
    ax.set(
        title=f"{letter}  {dataset} · {comparator}",
        xlabel="XenoComm percentile decile (10 = highest)",
        ylabel="Sampling seed",
    )

colors = {"XenoComm": "#D1495B", "CellChat": "#276FBF", "LIANA": "#4F8F65"}
sns.barplot(
    native_detection_counts,
    x="dataset",
    y="detected_ligands",
    hue="method",
    order=["PDX", "10x"],
    hue_order=list(colors),
    palette=colors,
    ax=axes[2, 0],
)
axes[2, 0].set(
    title="e  Ligands detected by each method", xlabel="", ylabel="Number of ligands"
)
axes[2, 0].legend(title=None)
for dataset in ("PDX", "10x"):
    for comparator in ("CellChat", "LIANA"):
        frame = topk_overlap.loc[
            topk_overlap.dataset.eq(dataset) & topk_overlap.comparator.eq(comparator)
        ].sort_values("k")
        axes[2, 1].plot(
            frame.k,
            frame.jaccard,
            marker="o",
            color=colors[comparator],
            linestyle="-" if dataset == "PDX" else "--",
            label=f"{dataset} – {comparator}",
        )
axes[2, 1].set(
    title="f  Top-k overlap with XenoComm",
    xlabel="Top-k ligands",
    ylabel="Jaccard overlap",
    xticks=TOP_K,
    ylim=(0, 1),
)
axes[2, 1].legend(title=None)
fig.savefig(OUTPUT_DIR / "si_figure_8.pdf", bbox_inches="tight")
fig.savefig(OUTPUT_DIR / "si_figure_8.png", dpi=200, bbox_inches="tight")
plt.show()